# Task 1 · Logistic regression (L1 / L2 / elastic net)

Payment-propensity models for Salud, pre-call (`paid_30d`) and post-call (`paid_after_call_30d`).
Instructions: `Phase 2/Task/1_logistic_regression.md`.

**Data rules:** print aggregates only (counts, rates, AUCs), never real rows. Real-data performance only via `real_eval()`.

In [ ]:
# ===================== SHARED SETUP: copy this whole block into the first cell, unchanged =====================
import json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings("ignore")

FAMILY = "logistic_regression"          # your model family (already filled in in your task file)
SEED = 42
# The repository root, found from wherever the notebook runs (Jupyter starts in the notebook's own folder).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "Phase 2").exists())
# The data file: in the repository root (where you save it), or in Phase 2/reports/ (the lead's copy).
DATA = next((p for p in (ROOT / "pre_training_data.xlsx", ROOT / "Phase 2" / "reports" / "pre_training_data.xlsx")
             if p.exists()), None)
assert DATA, f"pre_training_data.xlsx not found: save it in the repository root ({ROOT})"
OUT = ROOT / "Phase 2" / "notebooks" / FAMILY                    # your folder: notebook + committed, synthetic-only outputs
PRIVATE = ROOT / "Phase 2" / "reports" / "tasks" / FAMILY       # NOT committed (gitignored): anything derived from real rows
OUT.mkdir(parents=True, exist_ok=True); PRIVATE.mkdir(parents=True, exist_ok=True)

# ---------- data ----------
m = pd.read_excel(DATA)
BOOLS = ["label_usable", "post_call_row", "paid_30d", "paid_after_call_30d", "inbound", "spoke_with_agent",
         "collection_attempted", "promised_to_pay", "non_committal", "contested_or_refused", "pursued_insurance",
         "transferred", "action_shared_balance", "action_payment_plan", "action_financial_assistance",
         "action_statement_or_bill", "action_insurance", "asked_financial_assistance"]
for c in BOOLS:
    m[c] = m[c].astype(str).str.lower().map({"true": 1, "false": 0, "1": 1, "0": 0, "1.0": 1, "0.0": 0}).astype(float)
u = m[m.label_usable == 1]
SYN_TRAIN = u[(u.data_type == "synthetic") & (u.split == "train")]
SYN_TEST = u[(u.data_type == "synthetic") & (u.split == "test")]
REAL = u[u.data_type == "real"]
TARGET = {"pre": "paid_30d", "post": "paid_after_call_30d"}

def rows(df, target):
    """Pre-call model: every usable row. Post-call model: only calls without a payment on the call."""
    return df[df.post_call_row == 1] if target == "post" else df

# ---------- features ----------
PRE_ALL = ["n_accounts", "n_hospital_accounts", "n_physician_accounts", "placement_balance", "balance_at_t",
           "days_since_first_placement", "days_since_last_placement", "days_since_last_discharge", "n_payments_90d",
           "amount_paid_90d", "n_payments_365d", "amount_paid_365d", "n_payments_ever", "amount_paid_ever",
           "days_since_last_payment", "n_insurance_payments_365d", "amount_insurance_365d", "n_adjustments_365d",
           "amount_adjustments_365d", "paid_share_so_far", "last_payment_amount"]
CALL = ["inbound", "spoke_with_agent", "collection_attempted", "promised_to_pay", "non_committal", "contested_or_refused",
        "pursued_insurance", "transferred", "action_shared_balance", "action_payment_plan", "action_financial_assistance",
        "action_statement_or_bill", "action_insurance", "n_actions", "asked_financial_assistance", "talk_seconds",
        "caller_sentiment_overall", "caller_sentiment_end", "caller_sentiment_shift", "agent_sentiment_overall", "qa_score"]
CURRENT_PRE = ["n_accounts", "n_hospital_accounts", "n_physician_accounts", "balance_at_t", "days_since_first_placement",
               "days_since_last_placement", "n_payments_90d", "n_payments_365d", "days_since_last_payment",
               "n_insurance_payments_365d", "n_adjustments_365d"]
TIER12_PRE = ["n_payments_90d", "days_since_last_payment", "n_payments_365d", "paid_share_so_far", "last_payment_amount",
              "days_since_first_placement", "days_since_last_placement"]
SETS = {
    "pre": {
        "stable": ["n_payments_90d", "days_since_last_payment"],
        "top5": ["n_payments_90d", "days_since_last_payment", "n_payments_365d", "amount_adjustments_365d",
                 "days_since_first_placement"],
        "l1_one_se": ["n_payments_90d", "days_since_last_payment", "n_payments_365d", "amount_adjustments_365d",
                      "days_since_first_placement", "paid_share_so_far", "n_physician_accounts"],
        "tiers12": TIER12_PRE,
        "current": CURRENT_PRE,
        "all": PRE_ALL,
    },
    "post": {
        "stable": ["n_payments_90d", "days_since_last_payment", "action_payment_plan", "promised_to_pay",
                   "days_since_first_placement"],
        "top5": ["n_payments_90d", "days_since_last_payment", "action_payment_plan", "promised_to_pay",
                 "days_since_last_placement"],
        "l1_one_se": ["n_payments_90d", "days_since_last_payment", "action_payment_plan", "promised_to_pay",
                      "days_since_last_placement", "days_since_first_placement", "paid_share_so_far",
                      "amount_adjustments_365d", "spoke_with_agent", "n_payments_365d", "action_shared_balance",
                      "action_financial_assistance", "n_physician_accounts", "action_statement_or_bill"],
        "tiers12": TIER12_PRE + ["action_payment_plan", "promised_to_pay", "spoke_with_agent"],
        "current": CURRENT_PRE + CALL,
        "all": PRE_ALL + CALL,
    },
}
NEVER_USE = {"paid_30d", "paid_after_call_30d", "paid_amount_30d", "days_to_first_payment", "paid_fraction_30d",
             "paid_on_call", "payment_on_call", "label_usable", "label_reliable", "window_complete", "data_type",
             "split", "post_call_row", "sentiment_source", "qa_source", "guarantor_id", "call_id", "snapshot_id",
             "as_of", "agent", "tcn_record_id", "source", "ever_paid", "n_prior_contacts_90d", "n_prior_calls",
             "days_since_last_contact"}
assert not any(set(f) & NEVER_USE for s in SETS.values() for f in s.values()), "a forbidden column is in a feature set"
for c in PRE_ALL + CALL:
    m[c] = pd.to_numeric(m[c], errors="coerce")
SYN_TRAIN, SYN_TEST, REAL = (d.assign(**{c: pd.to_numeric(d[c], errors="coerce") for c in PRE_ALL + CALL})
                             for d in (SYN_TRAIN, SYN_TEST, REAL))

# ---------- preprocessing (put it INSIDE your pipeline, so it is fitted on training rows only) ----------
SKEWED = ["n_accounts", "n_hospital_accounts", "n_physician_accounts", "placement_balance", "balance_at_t",
          "days_since_first_placement", "days_since_last_placement", "days_since_last_discharge", "n_payments_90d",
          "amount_paid_90d", "n_payments_365d", "amount_paid_365d", "n_payments_ever", "amount_paid_ever",
          "days_since_last_payment", "n_insurance_payments_365d", "amount_insurance_365d", "n_adjustments_365d",
          "amount_adjustments_365d", "last_payment_amount", "n_actions", "talk_seconds"]

class Prep(BaseEstimator, TransformerMixin):
    """days_since_last_payment is empty when the patient never paid: fill with (train max + 1) and add a 0/1
    'never_paid' column. Any other gap -> 0. Optional log1p on skewed non-negative columns."""
    def __init__(self, log=True):
        self.log = log
    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.has_dslp_ = "days_since_last_payment" in X
        self.fill_ = (np.nanmax(X["days_since_last_payment"].to_numpy()) + 1) if self.has_dslp_ else None
        self.cols_ = list(X.columns) + (["never_paid"] if self.has_dslp_ else [])
        return self
    def transform(self, X):
        X = pd.DataFrame(X).copy()
        if self.has_dslp_:
            X["never_paid"] = X["days_since_last_payment"].isna().astype(float)
            X["days_since_last_payment"] = X["days_since_last_payment"].fillna(self.fill_)
        X = X.fillna(0.0)
        if self.log:
            for c in X.columns:
                if c in SKEWED:
                    X[c] = np.log1p(X[c].clip(lower=0))
        return X[self.cols_]
    def get_feature_names_out(self, input_features=None):
        return np.array(self.cols_)

# ---------- folds: everyone uses exactly these ----------
def folds(df, target, k=5, seed=SEED):
    """Grouped by patient: all calls of one patient are in the same fold."""
    y = df[TARGET[target]].astype(int)
    return list(StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed).split(df, y, df["guarantor_id"]))

# ---------- metrics ----------
def calibration(y, p):
    """Slope and intercept of the logistic recalibration (ideal: 1 and 0)."""
    z = np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6))).reshape(-1, 1)
    lr = LogisticRegression(C=1e6, max_iter=1000).fit(z, y)
    return float(lr.coef_[0][0]), float(lr.intercept_[0])

def metrics(y, p):
    y, p = np.asarray(y).astype(int), np.asarray(p, float)
    top = np.argsort(-p)[: max(1, int(round(0.2 * len(p))))]
    slope, icpt = calibration(y, p)
    return {"auc": roc_auc_score(y, p), "pr_auc": average_precision_score(y, p), "brier": brier_score_loss(y, p),
            "log_loss": log_loss(y, np.clip(p, 1e-6, 1 - 1e-6)), "calib_slope": slope, "calib_intercept": icpt,
            "top20_capture": y[top].sum() / max(y.sum(), 1), "mean_pred": p.mean(), "pay_rate": y.mean()}

def boot_ci(y, p, n=2000, seed=SEED):
    rng = np.random.default_rng(seed); y, p = np.asarray(y).astype(int), np.asarray(p, float); v = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() != y[i].max():
            v.append(roc_auc_score(y[i], p[i]))
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

def fit(model, X, y, w=None):
    """Fit with sample weights when the final step supports them (pipelines: last step must be named 'model')."""
    if w is None:
        return model.fit(X, y)
    try:
        return model.fit(X, y, model__sample_weight=w)
    except (TypeError, ValueError):
        return model.fit(X, y)          # estimator without sample_weight: real rows count once each

# ---------- 1) synthetic cross-validation: the main way to compare and choose ----------
def cv_eval(make_model, feats, target):
    """Grouped 5-fold CV on the synthetic training rows. Returns mean metrics, their SE, and out-of-fold predictions."""
    df = rows(SYN_TRAIN, target); y = df[TARGET[target]].astype(int).to_numpy()
    oof = np.full(len(df), np.nan); per = []; t0 = time.time()
    for tr, te in folds(df, target):
        mdl = fit(make_model(), df.iloc[tr][feats], y[tr])
        oof[te] = mdl.predict_proba(df.iloc[te][feats])[:, 1]
        per.append(metrics(y[te], oof[te]))
    per = pd.DataFrame(per)
    out = {f"cv_{k}": v for k, v in per.mean().items()}
    out["cv_auc_se"] = per["auc"].std(ddof=1) / np.sqrt(len(per))
    out["fit_seconds"] = round(time.time() - t0, 1)
    return out, pd.DataFrame({"call_id": df["call_id"].to_numpy(), "y": y, "p": oof})

# ---------- 2) real rows, measured fairly: repeated grouped cross-validation over the real patients ----------
REAL_LOOKS = {"n": 0}
def real_eval(make_model, feats, target, real_weight=0.0, repeats=10):
    """Score the real rows fairly. The real patients are split into 5 stratified groups (all calls of a patient
    together); each group is scored by a model trained on synthetic data + the OTHER real groups (each real row with
    weight real_weight; 0 = synthetic only). Repeated with 10 different splits. No real row is ever scored by a model
    that saw it.
    real_auc = AUC computed WITHIN each held-out group, averaged over the 50 groups. (Pooling the groups' predictions
    is not used for AUC: many real rows have identical features, and tiny differences between the groups' models
    would rank them at random. Leave-one-patient-out is not used either: it biases AUC downwards.)
    With real_weight = 0 there is one model, so the single-model AUC and its 95% bootstrap interval are also given.
    Every call counts as one 'look' at the real data."""
    REAL_LOOKS["n"] += 1
    syn = rows(SYN_TRAIN, target); real = rows(REAL, target).reset_index(drop=True)
    ys, yr = syn[TARGET[target]].astype(int).to_numpy(), real[TARGET[target]].astype(int).to_numpy()
    base = fit(make_model(), syn[feats], ys) if real_weight == 0 else None
    within, preds = [], []
    for r in range(repeats):
        p_r = np.full(len(real), np.nan)
        for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=SEED + r).split(real, yr, real["guarantor_id"]):
            if base is not None:
                mdl = base
            else:
                X = pd.concat([syn[feats], real.iloc[tr][feats]]); y = np.r_[ys, yr[tr]]
                w = np.r_[np.ones(len(ys)), np.full(len(tr), float(real_weight))]
                mdl = fit(make_model(), X, y, w)
            p_r[te] = mdl.predict_proba(real.iloc[te][feats])[:, 1]
            if yr[te].min() != yr[te].max():
                within.append(roc_auc_score(yr[te], p_r[te]))
        preds.append(p_r)
    p = np.mean(preds, axis=0)
    met = metrics(yr, p)
    out = {"real_auc": float(np.mean(within)), "real_auc_folds_sd": float(np.std(within, ddof=1))}
    out.update({f"real_{k}": v for k, v in met.items() if k != "auc"})
    if base is not None:   # one model for every real row: a plain AUC with a bootstrap interval is valid
        lo, hi = boot_ci(yr, p)
        out.update({"real_auc_single_model": met["auc"], "real_auc_lo": lo, "real_auc_hi": hi})
    out.update({"real_n": len(yr), "real_weight": real_weight, "real_looks_so_far": REAL_LOOKS["n"]})
    return out, pd.DataFrame({"call_id": real["call_id"].to_numpy(), "y": yr, "p": p})

# ---------- baseline everyone must beat: the rule "paid before, most recent first" ----------
def rule_scores(df):
    """Patients who paid before first, the most recent payers first. No training involved."""
    paid = df["days_since_last_payment"].notna().astype(float)
    return paid * 100000 - df["days_since_last_payment"].fillna(0)

def rule_real_auc(target, repeats=10):
    """The rule scored on the real rows exactly like real_eval (within-fold AUC, same splits): the number to beat."""
    real = rows(REAL, target).reset_index(drop=True); yr = real[TARGET[target]].astype(int).to_numpy()
    s = rule_scores(real).to_numpy(); within = []
    for r in range(repeats):
        for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=SEED + r).split(real, yr, real["guarantor_id"]):
            if yr[te].min() != yr[te].max():
                within.append(roc_auc_score(yr[te], s[te]))
    return float(np.mean(within))

# ---------- 3) final: fit once, score the synthetic TEST split once, save predictions ----------
def final_model(make_model, feats, target, real_weight=0.0, variant="", set_name="", params=None, notes=""):
    """Call this ONCE per target, for your chosen model. Fits on synthetic train (+ all real rows with real_weight),
    scores the synthetic test split, re-measures real fairly, logs a 'FINAL' row and saves predictions:
    synthetic out-of-fold + test predictions -> OUT (committed); real predictions -> PRIVATE (never committed)."""
    syn, test = rows(SYN_TRAIN, target), rows(SYN_TEST, target)
    ys, yt = syn[TARGET[target]].astype(int).to_numpy(), test[TARGET[target]].astype(int).to_numpy()
    if real_weight > 0:
        real = rows(REAL, target)
        X = pd.concat([syn[feats], real[feats]]); y = np.r_[ys, real[TARGET[target]].astype(int).to_numpy()]
        w = np.r_[np.ones(len(ys)), np.full(len(real), float(real_weight))]
        mdl = fit(make_model(), X, y, w)
    else:
        mdl = fit(make_model(), syn[feats], ys)
    p_test = mdl.predict_proba(test[feats])[:, 1]
    st = {f"test_{k}": v for k, v in metrics(yt, p_test).items()}
    st["test_auc_lo"], st["test_auc_hi"] = boot_ci(yt, p_test)
    st["test_rule_auc"] = roc_auc_score(yt, rule_scores(test))
    cv, oof = cv_eval(make_model, feats, target)
    real_m, real_p = real_eval(make_model, feats, target, real_weight)
    real_m["real_rule_auc"] = rule_real_auc(target)
    log_result("FINAL " + variant, target, set_name, feats, params or {}, cv, real_m, st, notes)
    pd.concat([oof.assign(kind="synthetic_oof"), pd.DataFrame({"call_id": test["call_id"].to_numpy(), "y": yt,
               "p": p_test, "kind": "synthetic_test"})]).assign(family=FAMILY, target=target, variant=variant,
               feature_set=set_name).to_csv(OUT / f"synthetic_predictions_{target}.csv", index=False)
    real_p.assign(family=FAMILY, target=target, variant=variant, feature_set=set_name).to_csv(
        PRIVATE / f"real_predictions_{target}.csv", index=False)
    return mdl, {**cv, **real_m, **st}

# ---------- logging: one row per experiment ----------
RESULTS = OUT / "results.csv"
def log_result(variant, target, set_name, feats, params, cv, real=None, synthetic_test=None, notes=""):
    row = {"family": FAMILY, "variant": variant, "target": target, "feature_set": set_name, "n_features": len(feats),
           "params": json.dumps(params, default=str), **cv, **(real or {}), **(synthetic_test or {}), "notes": notes,
           "logged_at": pd.Timestamp.now().isoformat(timespec="seconds")}
    df = pd.DataFrame([row])
    df.to_csv(RESULTS, mode="a", header=not RESULTS.exists(), index=False)
    return row

print(f"synthetic train: pre {len(rows(SYN_TRAIN, 'pre')):,} / post {len(rows(SYN_TRAIN, 'post')):,} rows; "
      f"synthetic test: pre {len(rows(SYN_TEST, 'pre')):,} / post {len(rows(SYN_TEST, 'post')):,}; "
      f"real: pre {len(rows(REAL, 'pre'))} / post {len(rows(REAL, 'post'))}")
# =========================================== END OF SHARED SETUP ===========================================

## Data check (section 3)
Expected: shape `(8833, 76)`, fingerprint `acd75c39fe5bfa98`, usable rows 58 / 1,702 / 6,647.

In [ ]:
import hashlib
m = pd.read_excel(ROOT / "pre_training_data.xlsx")
key = m.sort_values("call_id")[["call_id", "split", "paid_30d", "paid_after_call_30d"]].astype(str).agg("|".join, axis=1)
print(m.shape, hashlib.sha256("\n".join(key).encode()).hexdigest()[:16])
print(m.groupby(["data_type", "split"]).agg(rows=("call_id", "size"), usable=("label_usable", "sum")))
del m, key

## 1 · Screen (section 9, step 1)
Every section-8 variant with default settings, on all six sets × both targets, with `cv_eval`. Log every run.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, QuantileTransformer, SplineTransformer

# Start a fresh results.csv so the notebook can be rerun top to bottom without duplicate rows.
if RESULTS.exists():
    RESULTS.unlink()

# Keep every logged row in memory too. The shared log_result writes the CSV header only once, so rows with extra
# columns (from real_eval / final_model) don't line up; section 7 rewrites results.csv from this list.
# final_model calls log_result by name, so this wrapper also captures the FINAL rows. The shared block is unchanged.
LOG = []
_shared_log_result = log_result
def log_result(*args, **kwargs):
    row = _shared_log_result(*args, **kwargs)
    LOG.append(row)
    return row

def lr_pipe(model, log=True, scaler="standard", spline_knots=None):
    """Prep -> [spline] -> scaler -> logistic regression. The last step is named 'model' (needed for sample weights)."""
    steps = [("prep", Prep(log=log))]
    if spline_knots:
        steps.append(("spline", SplineTransformer(n_knots=spline_knots, degree=3)))
    steps.append(("scale", QuantileTransformer(output_distribution="normal", random_state=SEED)
                  if scaler == "quantile" else StandardScaler()))
    steps.append(("model", model))
    return Pipeline(steps)

# Section 8 variants at sensible default settings (C = 1, l1_ratio = 0.5, 4 spline knots).
SCREEN = {
    "no penalty":         (lambda: lr_pipe(LogisticRegression(C=1e6, max_iter=5000)), {"C": 1e6}),
    "L2":                 (lambda: lr_pipe(LogisticRegression(penalty="l2", C=1.0, max_iter=5000)), {"C": 1.0}),
    "L1":                 (lambda: lr_pipe(LogisticRegression(penalty="l1", C=1.0, solver="liblinear", max_iter=5000,
                                                              random_state=SEED)), {"C": 1.0}),
    "elastic net":        (lambda: lr_pipe(LogisticRegression(penalty="elasticnet", C=1.0, l1_ratio=0.5, solver="saga",
                                                              max_iter=5000, random_state=SEED)), {"C": 1.0, "l1_ratio": 0.5}),
    "L2, no log":         (lambda: lr_pipe(LogisticRegression(penalty="l2", C=1.0, max_iter=5000), log=False), {"C": 1.0, "log": False}),
    "L2, no log, quantile": (lambda: lr_pipe(LogisticRegression(penalty="l2", C=1.0, max_iter=5000), log=False,
                                             scaler="quantile"), {"C": 1.0, "log": False, "scaler": "quantile"}),
    "L2 + splines":       (lambda: lr_pipe(LogisticRegression(penalty="l2", C=1.0, max_iter=5000), spline_knots=4),
                           {"C": 1.0, "n_knots": 4}),
    "L2, balanced":       (lambda: lr_pipe(LogisticRegression(penalty="l2", C=1.0, max_iter=5000, class_weight="balanced")),
                           {"C": 1.0, "class_weight": "balanced"}),
}

screen_rows = []
t0 = time.time()
for target in ["pre", "post"]:
    for set_name, feats in SETS[target].items():
        for variant, (make, params) in SCREEN.items():
            cv, _ = cv_eval(make, feats, target)
            log_result(f"screen: {variant}", target, set_name, feats, params, cv)
            screen_rows.append({"target": target, "set": set_name, "n": len(feats), "variant": variant,
                                "cv_auc": cv["cv_auc"], "cv_auc_se": cv["cv_auc_se"], "cv_brier": cv["cv_brier"]})
screen = pd.DataFrame(screen_rows)
print(f"{len(screen)} screening runs in {time.time() - t0:.0f}s (synthetic train only; real looks so far: {REAL_LOOKS['n']})")


In [ ]:
for target in ["pre", "post"]:
    t = screen[screen.target == target].pivot(index="variant", columns="set", values="cv_auc")
    t = t[list(SETS[target])]
    print(f"\n{target}-call · CV AUC (synthetic train, grouped 5-fold)")
    print(t.round(3).to_string())
    best = screen[screen.target == target].sort_values("cv_auc", ascending=False).head(5)
    print("top 5:"); print(best[["variant", "set", "n", "cv_auc", "cv_auc_se", "cv_brier"]].round(4).to_string(index=False))


## 2 · Tune (section 9, step 2)
Top 3 sets per target from the screen: pre `tiers12`, `all`, `current`; post `tiers12`, `stable`, `l1_one_se`.
About 40 settings per set and target: L2 × 13 `C`, L1 × 7 `C`, elastic net × 3 `l1_ratio` × 3 `C`, L2 + splines × 4 knot counts × 3 `C`. Synthetic train only.

In [ ]:
TUNE_SETS = {"pre": ["tiers12", "all", "current"], "post": ["tiers12", "stable", "l1_one_se"]}
C_GRID = np.logspace(-4, 2, 13)

def tune_grid():
    """(variant, params, make_model) for every setting tried; about 40 per set and target."""
    g = []
    for C in C_GRID:
        g.append(("L2", {"C": C}, lambda C=C: lr_pipe(LogisticRegression(penalty="l2", C=C, max_iter=5000))))
    for C in C_GRID[::2]:
        g.append(("L1", {"C": C}, lambda C=C: lr_pipe(LogisticRegression(
            penalty="l1", C=C, solver="liblinear", max_iter=5000, random_state=SEED))))
    for r in [0.2, 0.5, 0.8]:
        for C in [0.01, 0.1, 1.0]:
            g.append(("elastic net", {"C": C, "l1_ratio": r}, lambda C=C, r=r: lr_pipe(LogisticRegression(
                penalty="elasticnet", C=C, l1_ratio=r, solver="saga", max_iter=5000, random_state=SEED))))
    for k in [3, 4, 5, 6]:
        for C in [0.01, 0.1, 1.0]:
            g.append(("L2 + splines", {"C": C, "n_knots": k}, lambda C=C, k=k: lr_pipe(
                LogisticRegression(penalty="l2", C=C, max_iter=5000), spline_knots=k)))
    return g

tune_rows = []
t0 = time.time()
for target, sets in TUNE_SETS.items():
    for set_name in sets:
        feats = SETS[target][set_name]
        for variant, params, make in tune_grid():
            cv, _ = cv_eval(make, feats, target)
            log_result(f"tune: {variant}", target, set_name, feats, params, cv)
            tune_rows.append({"target": target, "set": set_name, "n": len(feats), "variant": variant,
                              "params": json.dumps({k: (round(v, 6) if isinstance(v, float) else v) for k, v in params.items()}),
                              "cv_auc": cv["cv_auc"], "cv_auc_se": cv["cv_auc_se"], "cv_brier": cv["cv_brier"],
                              "cv_calibration_slope": cv.get("cv_calibration_slope", np.nan)})
tuned = pd.DataFrame(tune_rows)
print(f"{len(tuned)} tuning runs in {time.time() - t0:.0f}s ({len(tune_grid())} settings per set and target; "
      f"real looks so far: {REAL_LOOKS['n']})")


In [ ]:
for target in ["pre", "post"]:
    t = tuned[tuned.target == target]
    best = t.loc[t.cv_auc.idxmax()]
    print(f"\n{target}-call · best CV AUC {best.cv_auc:.4f} ± {best.cv_auc_se:.4f} ({best.variant}, {best.set}, {best.params})")
    print("best per variant and set:")
    per = t.loc[t.groupby(["variant", "set"]).cv_auc.idxmax()].sort_values("cv_auc", ascending=False)
    print(per[["variant", "set", "n", "params", "cv_auc", "cv_auc_se", "cv_brier"]].round(4).to_string(index=False))
    within = t[t.cv_auc >= best.cv_auc - best.cv_auc_se]
    print(f"settings within one SE of the best: {len(within)} of {len(t)}")


## 3 · Use the real rows (section 9, step 3)
4 candidates per target × real weights 0 / 1 / 5 / 20, measured only with `real_eval` (each call is one look). Real AUCs carry about ±0.10–0.15 uncertainty; only aggregates are printed.

In [ ]:
def spline_l2(C, k): return lambda: lr_pipe(LogisticRegression(penalty="l2", C=C, max_iter=5000), spline_knots=k)
def plain_l2(C):     return lambda: lr_pipe(LogisticRegression(penalty="l2", C=C, max_iter=5000))
def plain_l1(C):     return lambda: lr_pipe(LogisticRegression(penalty="l1", C=C, solver="liblinear", max_iter=5000,
                                                               random_state=SEED))
CANDIDATES = {
    "pre": [("L2 + splines", "all",     {"C": 0.01, "n_knots": 3}, spline_l2(0.01, 3)),
            ("L2 + splines", "tiers12", {"C": 0.01, "n_knots": 5}, spline_l2(0.01, 5)),
            ("L2",           "all",     {"C": float(C_GRID[11])},  plain_l2(float(C_GRID[11]))),
            ("L1",           "tiers12", {"C": 0.1},                plain_l1(0.1))],
    "post": [("L2 + splines", "tiers12",   {"C": 0.01, "n_knots": 5}, spline_l2(0.01, 5)),
             ("L2 + splines", "stable",    {"C": 0.1, "n_knots": 5},  spline_l2(0.1, 5)),
             ("L2",           "tiers12",   {"C": float(C_GRID[5])},   plain_l2(float(C_GRID[5]))),
             ("L2",           "l1_one_se", {"C": float(C_GRID[5])},   plain_l2(float(C_GRID[5])))],
}
WEIGHTS = [0, 1, 5, 20]

real_rows = []
for target, cands in CANDIDATES.items():
    for variant, set_name, params, make in cands:
        feats = SETS[target][set_name]
        cv, _ = cv_eval(make, feats, target)
        for w in WEIGHTS:
            real, _ = real_eval(make, feats, target, real_weight=w)
            log_result(f"real check: {variant}", target, set_name, feats, params | {"real_weight": w}, cv, real)
            real_rows.append({"target": target, "variant": variant, "set": set_name, "n": len(feats),
                              "params": json.dumps(params), "real_weight": w, "cv_auc": cv["cv_auc"],
                              "cv_auc_se": cv["cv_auc_se"], "real_auc": real["real_auc"],
                              "real_auc_folds_sd": real["real_auc_folds_sd"], "real_brier": real["real_brier"]})
real_check = pd.DataFrame(real_rows)
RULE_REAL = {t: rule_real_auc(t) for t in ["pre", "post"]}
print(f"real looks so far: {REAL_LOOKS['n']}; rule baseline (real): pre {RULE_REAL['pre']:.3f} / post {RULE_REAL['post']:.3f}")
for target in ["pre", "post"]:
    t = real_check[real_check.target == target]
    piv = t.pivot_table(index=["variant", "set", "n", "params", "cv_auc"], columns="real_weight", values="real_auc")
    print(f"\n{target}-call · real AUC (within-group, by real weight)")
    print(piv.round(3).to_string())


## 4 · Pick one final model per target (section 9, step 4)
- **Pre-call:** L2 + splines, `all` (21), 3 knots, C = 0.01, real weight 1. Within one SE of the best CV (it *is* the best, 0.717); highest real AUC (0.873; weight 5 ties at 0.872, weight 1 relies less on real rows; weight 0 is 0.852, just outside 0.02). Plain L2 on real data is far lower (0.73–0.76), so the spline curves earn their place.
- **Post-call:** plain L2, `tiers12` (10), C = 0.0316, real weight 0. All four candidates are within one SE on CV (≥ 0.798). The highest real AUC is L2 + splines at weight 5 (0.771); plain L2 at weight 0 (0.757) is within 0.02, so the simpler model wins: 10 coefficients instead of ~70 spline terms, one odds ratio per feature, no real rows in training. Top-20% capture is the same (39.6% vs 39.8% in CV).

## 5 · Finish (section 9, step 5)
`final_model` once per target: scores the synthetic test split once, re-measures real data fairly, logs the `FINAL` rows and saves prediction files.

In [ ]:
FINAL = {
    "pre":  dict(variant="L2 + splines", set_name="all", params={"C": 0.01, "n_knots": 3, "real_weight": 1},
                 make=spline_l2(0.01, 3), real_weight=1),
    "post": dict(variant="L2", set_name="tiers12", params={"C": float(C_GRID[5]), "real_weight": 0},
                 make=plain_l2(float(C_GRID[5])), real_weight=0),
}
final_models, final_metrics = {}, {}
for target, f in FINAL.items():
    feats = SETS[target][f["set_name"]]
    mdl, met = final_model(f["make"], feats, target, real_weight=f["real_weight"], variant=f["variant"],
                           set_name=f["set_name"], params=f["params"])
    final_models[target], final_metrics[target] = mdl, met

keys = ["cv_auc", "cv_auc_se", "real_auc", "real_rule_auc", "test_auc", "test_auc_lo", "test_auc_hi", "test_rule_auc",
        "test_pr_auc", "test_brier", "test_log_loss", "test_calib_slope", "test_top20_capture"]
print(pd.DataFrame(final_metrics).loc[[k for k in keys if k in final_metrics["pre"]]].round(4).to_string())
print(f"real looks in this run: {REAL_LOOKS['n']}")


## 6 · Feature notes for the report
Post-call (plain L2): standardized coefficients and odds ratios per one SD. Pre-call (splines): no single coefficient per feature, so the effect is shown as the change in predicted log-odds when one feature moves from its 10th to its 90th percentile on synthetic train, others at their medians. Near-duplicate payment features share weight; don't read them one by one.

In [ ]:
post = final_models["post"]
names = post.named_steps["prep"].get_feature_names_out()
c = post.named_steps["model"].coef_[0]
post_coef = pd.DataFrame({"feature": names, "coef_per_sd": c, "odds_ratio_per_sd": np.exp(c)})
post_coef["direction"] = np.where(c > 0, "more likely to pay", "less likely to pay")
post_coef = post_coef.reindex(post_coef.coef_per_sd.abs().sort_values(ascending=False).index)
print("post-call · plain L2"); print(post_coef.round(3).to_string(index=False))

pre = final_models["pre"]; feats = SETS["pre"]["all"]
base = rows(SYN_TRAIN, "pre")[feats]
med = base.median(numeric_only=True).to_frame().T
logit = lambda X: np.log(pre.predict_proba(X)[:, 1] / (1 - pre.predict_proba(X)[:, 1]))
eff = []
for f in feats:
    lo, hi = med.copy(), med.copy()
    lo[f], hi[f] = base[f].quantile(0.10), base[f].quantile(0.90)
    d = float(logit(hi[feats])[0] - logit(lo[feats])[0])
    eff.append({"feature": f, "logodds_p10_to_p90": d, "odds_ratio_p10_to_p90": np.exp(d)})
pre_eff = pd.DataFrame(eff).sort_values("logodds_p10_to_p90", key=abs, ascending=False)
print("\npre-call · L2 + splines (synthetic train, p10 -> p90)"); print(pre_eff.round(3).to_string(index=False))


## 7 · Tidy results.csv
The shared `log_result` appends rows but writes the header only once, so rows logged with `real_eval` or `final_model` metrics have more fields than the header and the file can't be read as one table. This rewrites `results.csv` from the in-memory log with every column (same rows, nothing dropped).

In [ ]:
res = pd.DataFrame(LOG)
res.to_csv(RESULTS, index=False)
check = pd.read_csv(RESULTS)
print(f"results.csv: {len(check)} rows, {check.shape[1]} columns; FINAL rows: {check.variant.str.startswith('FINAL').sum()}")
print(check.groupby(check.variant.str.split(":").str[0].str.split(" ").str[0]).size().to_string())
